# Phase 2: Severity Tracker (UPDRS Estimation)
This notebook trains a Random Forest Regressor to predict the **motor_UPDRS score** from acoustic voice features.

**Dataset:** `LailaQadirMusib.csv` (5875 recordings from 42 patients)
- `motor_UPDRS`: Clinician-scored motor Unified Parkinson's Disease Rating Scale (range: ~5 – 40)

**Features used (16 acoustic columns):**  
Jitter(%), Jitter(Abs), Jitter:RAP, Jitter:PPQ5, Jitter:DDP,  
Shimmer, Shimmer(dB), Shimmer:APQ3, Shimmer:APQ5, Shimmer:APQ11, Shimmer:DDA,  
NHR, HNR, RPDE, DFA, PPE

> **Important:** This exact feature set must match `backend/utils.py` (indices 3–17 and 21 of the voice feature vector) and `train_models.py`. The columns `age`, `sex`, `test_time`, `total_UPDRS` are deliberately excluded.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load Dataset (path relative to notebooks/ folder)
df2 = pd.read_csv('../data/LailaQadirMusib.csv')
print('Dataset shape:', df2.shape)
print('Columns:', list(df2.columns))

# 2. Define Features (X) and Target (y)
# We predict 'motor_UPDRS' using ONLY acoustic features.
# Drop: subject# (ID), age, sex, test_time (metadata), total_UPDRS (leakage), motor_UPDRS (target)
# This leaves exactly 16 acoustic features — MUST match train_models.py and backend/utils.py
X_reg = df2.drop(['subject#', 'age', 'sex', 'test_time', 'motor_UPDRS', 'total_UPDRS'], axis=1)
y_reg = df2['motor_UPDRS']

print(f'\nFeatures used ({len(X_reg.columns)} columns):')
for i, col in enumerate(X_reg.columns):
    print(f'  {i}: {col}')
print(f'\nTarget range: {y_reg.min():.2f} to {y_reg.max():.2f}')

# 3. Split the data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 4. Scale the Features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg.values)  # .values to strip column names
X_test_reg_scaled  = scaler_reg.transform(X_test_reg.values)

# 5. Train the Random Forest Regressor (same settings as train_models.py)
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_reg_scaled, y_train_reg)

# 6. Test the Model
reg_preds = rf_regressor.predict(X_test_reg_scaled)

# 7. Evaluate the Results
mae = mean_absolute_error(y_test_reg, reg_preds)
r2  = r2_score(y_test_reg, reg_preds)
print("\n--- MODULE 2: SEVERITY TRACKER (Random Forest Regressor) ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}  ← avg error in UPDRS points")
print(f"R-squared Score:           {r2 * 100:.2f}%  ← how much variance is explained")
print("\nNote: The saved severity_model.pkl was trained with these exact same settings.")